In [ ]:
import gc
from pathlib import Path

from shapely.geometry import box

import cyanomembranes as cm

(OUT := Path("output")).mkdir(exist_ok=True)
import sys
from pathlib import Path

# Add the folder that CONTAINS membrane_analysis/ to the path
sys.path.insert(0, str(Path("..").resolve()))  # adjust if needed


In [ ]:
from Bio.PDB import PDBParser, NeighborSearch
import numpy as np

parser = PDBParser(QUIET=True)
structure = parser.get_structure("protein", "../tutorial_plants/data_plants/6RQF-Cyt-spinach.pdb")

In [ ]:
model = structure[0]

# Define which chains you want to search
TARGET_CHAINS = {"A", "I"}  # ← change these to whatever chains you want

# Collect PL9 only from target chains
ligands = []
for chain in model:
    if chain.id in TARGET_CHAINS:
        for residue in chain:
            if residue.resname == "PL9":
                ligands.append(residue)
                print(f"Found PL9 in chain {chain.id}, residue {residue.id}")

# Build neighbor search over all atoms
all_atoms = list(structure.get_atoms())
ns = NeighborSearch(all_atoms)
CUTOFF = 5.0

# Get one pocket per ligand
all_pockets = {}
for ligand in ligands:
    chain_id = ligand.get_parent().id
    ligand_atoms = list(ligand.get_atoms())

    pocket_residues = set()
    for ligand_atom in ligand_atoms:
        nearby_atoms = ns.search(ligand_atom.coord, CUTOFF)
        for atom in nearby_atoms:
            res = atom.get_parent()
            if res != ligand:
                pocket_residues.add(res)

    all_pockets[chain_id] = pocket_residues
    print(f"Pocket in chain {chain_id}: {len(pocket_residues)} residues")

In [ ]:
all_pockets

In [ ]:
 #--- 5. Report binding pocket residues ---
print(f"{'Chain':<8}{'ResName':<10}{'ResNum':<10}")
print("-" * 28)

for res in sorted(pocket_residues, key=lambda r: (r.get_parent().id, r.id[1])):
    chain_id = res.get_parent().id
    print(f"{chain_id:<8}{res.resname:<10}{res.id[1]:<10}")

print(f"\nTotal pocket residues: {len(pocket_residues)}")

# --- 6. (Optional) Get per-contact distances ---
print("\nClosest contact per pocket residue:")
print(f"{'Chain':<8}{'ResName':<10}{'ResNum':<10}{'Min Dist (Å)':<14}{'Ligand Atom':<14}{'Res Atom'}")
print("-" * 65)

contact_info = []
for res in pocket_residues:
    min_dist = float("inf")
    closest_pair = (None, None)
    for l_atom in ligand_atoms:
        for r_atom in res.get_atoms():
            dist = np.linalg.norm(l_atom.coord - r_atom.coord)
            if dist < min_dist:
                min_dist = dist
                closest_pair = (l_atom.name, r_atom.name)
    contact_info.append((res, min_dist, closest_pair))

for res, dist, (la, ra) in sorted(contact_info, key=lambda x: x[1]):
    chain_id = res.get_parent().id
    print(f"{chain_id:<8}{res.resname:<10}{res.id[1]:<10}{dist:<14.2f}{la:<14}{ra}")


In [ ]:
all_pockets.keys()

In [ ]:
atom_coords = []

for c in ["I"]:
    chain = all_pockets[c]
    for res in chain:
        for atom in res.get_atoms():
            atom_coords.append(atom.coord[:2])

atom_coords = np.array(atom_coords)

In [ ]:
PDB_FILES = [
    "6RQF-Cyt-spinach.pdb"
]

DATA = Path("../tutorial_plants/data_plants")
(OUT := Path("../tutorial_plants/outputs")).mkdir(exist_ok=True)
(PIC_DIR :=  (OUT/"pictures_membranes")).mkdir(exist_ok=True)

proteins = cm.pdb_utils.process_proteins(DATA, OUT / "protein_shadows", PDB_FILES)


In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots()
psii = proteins["6RQF-Cyt-spinach"]["polygon"][0]
ax.plot(*psii.exterior.xy)
ax.set_aspect("equal")
ax.scatter(atom_coords[:,0], atom_coords[:,1], s=1)


In [ ]:
from shapely.geometry import Point
from shapely.ops import nearest_points

def project_on_nperimeter(polygon, coords):
    cx, cy = psii.centroid.x, psii.centroid.y
    centroid = np.array([cx, cy])
    projected = []
    for pt in atom_coords:
        p = Point(pt)
        nearest = nearest_points(psii.boundary, p)[0]
        projected.append((nearest.x, nearest.y))

    return np.array(projected)

projected = project_on_nperimeter(psii, atom_coords)
cx, cy = psii.centroid.x, psii.centroid.y
angles = np.arctan2(projected[:, 1] - cy,
                    projected[:, 0] - cx)  # in [-π, π]

# --- Mean angle (circular mean to handle wrap-around at ±π) ---
mean_angle = np.arctan2(np.sin(angles).mean(),
                        np.cos(angles).mean())

# --- Angular distance from mean ---
angle_diff = np.abs(np.arctan2(np.sin(angles - mean_angle),
                               np.cos(angles - mean_angle)))

# --- Keep only points within a threshold (e.g. 45 degrees) ---
THRESHOLD = np.deg2rad(45)  # adjust if needed
mask = angle_diff < THRESHOLD
projected_filtered = projected[mask]

In [ ]:
center = projected_filtered.mean(axis=0)

# Snap center onto the perimeter
center_on_perimeter = nearest_points(psii.boundary, Point(center))[0]

radius = 5

# Shapely circle via buffer — center is now exactly on the boundary
shapely_circle = Point(center_on_perimeter).buffer(radius)

fig, ax = plt.subplots()
psii = proteins["6RQF-Cyt-spinach"]["polygon"][0]
ax.plot(*psii.exterior.xy)
ax.set_aspect("equal")
ax.scatter(atom_coords[:, 0], atom_coords[:, 1], s=1)
ax.scatter(projected_filtered[:, 0], projected_filtered[:, 1], s=5)
ax.plot(*shapely_circle.exterior.xy)
